# Injury severity: gamma GLM

Gamma GLM (log link) for days missed per injury. Predictors: severity tier, position (attacker / defender / goalkeeper vs. midfielder), age (centred, + age²), `log(1 + prior_injuries)`, `reinjury_60d`, `prior_same_part`.

In [1]:
import pandas as pd
import severity as sev

data = sev.build_dataset()
print(f"{len(data):,} injuries, {data.pid.nunique():,} players")

163,414 injuries, 34,429 players


## ▶ GLM fit — gamma GLM (log link)

Fit on all injuries.

In [2]:
# ===================== GLM FIT =====================
glm = sev.TieredGammaGLM().fit(data)

## Coefficients

Coefficients are on the log scale. Reference: the tier containing Hamstring (unspecified) (no row in the table), midfielder.

In [3]:
coefs = pd.DataFrame({"coef": glm.res.params.to_numpy(), "p_value": glm.res.pvalues.to_numpy()},
                     index=sev.tidy_terms(glm.res.params.index)).round(4)
coefs.index.name = "variable"
coefs

,coef,p_value
variable,,
Intercept,3.9557,0.0000
severity_tier = Tier 1,-1.3462,0.0000
severity_tier = Tier 2,-0.7536,0.0000
severity_tier = Tier 3,-0.6300,0.0000
severity_tier = Tier 4,-0.4838,0.0000
severity_tier = Tier 5,-0.2895,0.0000
severity_tier = Tier 6,-0.1937,0.0000
severity_tier = Tier 7,-0.1051,0.0000
severity_tier = Tier 9,0.1999,0.0000


## Severity tiers

Tiers are rating classes merged until neighbours differ at p < 0.05; Tier 1 = shortest absences.

In [4]:
tiers = pd.Series(glm.tier_of).groupby(pd.Series(glm.tier_of)).apply(lambda s: "; ".join(sorted(s.index)))
tiers = tiers.reindex(glm.tier_names).rename("rating classes").to_frame()
tiers.index.name = "tier"
with pd.option_context("display.max_colwidth", None):
    display(tiers)

,rating classes
tier,
Tier 1,Illness (unspecified)
Tier 2,Head - Concussion; Knee - Knock / bruise; Trunk - Knock / bruise; Unknown - Knock / bruise; Unknown - Strain / muscle
Tier 3,Foot - Knock / bruise; Knock / bruise - other sites
Tier 4,Ankle - Ligament / joint; Head (unspecified); Muscle (unspecified) - Strain / muscle; Strain / muscle - other sites; Thigh (unspecified); Thigh - Knock / bruise
Tier 5,Calf (unspecified); Calf - Strain / muscle; Hamstring - Strain / muscle; Head - Fracture
Tier 6,Groin (unspecified); Muscle (unspecified) - Tear / rupture; Unknown (unspecified)
Tier 7,Arm / hand (unspecified); Back / neck (unspecified); Trunk - Fracture
Tier 8,Ankle (unspecified); Arm / hand - Fracture; Calf - Tear / rupture; Foot (unspecified); Groin - Tear / rupture; Hamstring (unspecified); Other (rare); Trunk - Strain / muscle; Unknown - Tendon / inflammation
Tier 9,Achilles - Tendon / inflammation; Hip (unspecified); Unspecified injury - other sites
